# Testing a Sequence of Risk Forecasts

## Forecasts Made Before the Held-Out Losses

For forecast date \(t\), define the exception indicator

$$
I_t=\mathbf{1}\{L_t>q_t\}.
$$

The loss observed on date \(t\) is \(L_t\), the quantile forecast made before that loss is \(q_t\), and \(\mathbf{1}\{\cdot\}\) equals one when its condition is true. At tail probability \(a\), a correctly calibrated quantile forecast has \(\Pr(I_t=1)=a\). This statement concerns quantile exceptions. It does not test the size of losses beyond the quantile or validate expected shortfall.

In [1]:
from pathlib import Path
import hashlib
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display


def display(value):
    module = value.__class__.__module__
    if module.startswith("pandas") and hasattr(value, "_repr_html_"):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + "</div>"))
    return _display(value)
from scipy.stats import genpareto, norm


BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset

import RiskLabAI
from RiskLabAI.backtest.backtest_overfitting_simulation import expected_shortfall
from RiskLabAI.backtest.robust_statistics import conditional_expected_drawdown
from RiskLabAI.backtest.tail_risk import fit_pot_tail_risk, generalized_pareto_tail_risk, quantile_exception_tests

assert RiskLabAI.__version__ == "3.2.0"

DATA_PATH = ensure_dataset(BOOK_ROOT, "fred_nasdaq")
FIGURES = BOOK_ROOT / "outputs" / "figures" / "21_tail_risk"
TABLES = BOOK_ROOT / "outputs" / "tables" / "21_tail_risk"
FIGURE_DATA = BOOK_ROOT / "outputs" / "figure_data" / "21_tail_risk"
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "background": "#FFFFFF",
    "axes": "#FFFFFF",
    "text": "#1B365D",
    "cyan": "#0072B2",
    "amber": "#C76D00",
    "green": "#00866A",
    "rose": "#B83B3B",
    "violet": "#A03A78",
    "grid": "#D9DEE5",
}
plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 11.5,
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor": PALETTE["axes"],
    "axes.edgecolor": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "axes.titlecolor": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "text.color": PALETTE["text"],
    "grid.color": PALETTE["grid"],
    "grid.linestyle": ":",
    "legend.facecolor": PALETTE["axes"],
    "legend.edgecolor": PALETTE["grid"],
    "savefig.facecolor": PALETTE["background"],
    "savefig.transparent": False,
    "savefig.dpi": 300,
})


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()


def rendered_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (bool, np.bool_)):
        return "Pass" if bool(value) else "Fail"
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).date().isoformat()
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):
        number = float(value)
        if np.isposinf(number):
            return "Infinite"
        if np.isneginf(number):
            return "Negative infinite"
        if number == 0:
            return "0"
        if number.is_integer():
            return f"{int(number):,}"
        magnitude = abs(number)
        if magnitude < 0.0001 or magnitude >= 1000000:
            return f"{number:.6g}"
        return f"{number:.6f}".rstrip("0").rstrip(".")
    return str(value)


def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))


def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position="htbp")
    if fit:
        latex = latex.replace("\\begin{tabular}", "\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def finish_axis(axis, grid=True):
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)


def save_figure(figure, stem):
    figure.savefig(FIGURES / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    figure.savefig(FIGURES / f"{stem}.pdf", bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)


def load_public_data():
    expected_hash = "2E988948FFCB108DA4902DFEDAA5DF756E7E102DC523A99A8916A0533291F2D7"
    assert DATA_PATH.is_file()
    assert sha256(DATA_PATH) == expected_hash
    raw = pd.read_csv(DATA_PATH)
    raw["Date"] = pd.to_datetime(raw["observation_date"])
    raw["NASDAQ Composite"] = pd.to_numeric(raw["NASDAQCOM"], errors="coerce")
    data = raw.dropna(subset=["NASDAQ Composite"]).sort_values("Date").reset_index(drop=True)
    data["Daily log return (%)"] = 100 * np.log(data["NASDAQ Composite"] / data["NASDAQ Composite"].shift(1))
    data["Loss (%)"] = -data["Daily log return (%)"]
    data["Simple return"] = np.expm1(data["Daily log return (%)"] / 100)
    return data


def split_samples(data):
    returns = data.dropna(subset=["Loss (%)"]).copy()
    design = returns[returns["Date"] <= pd.Timestamp("2014-12-31")].copy()
    held = returns[returns["Date"] >= pd.Timestamp("2015-01-01")].copy()
    assert len(design) == 5287
    assert len(held) == 2767
    return returns, design, held


def empirical_upper_tail_mean(losses, probability):
    values = np.sort(np.asarray(losses, dtype=float))
    count = (1 - probability) * len(values)
    full = int(np.floor(count))
    total = values[len(values) - full:].sum() if full > 0 else 0.0
    fraction = count - full
    if fraction > 0:
        total += fraction * values[len(values) - full - 1]
    return float(total / count)


def rolling_max_drawdowns(simple_returns, horizon):
    wealth = np.cumprod(1 + np.asarray(simple_returns, dtype=float))
    if len(wealth) < horizon or horizon < 2:
        drawdown = 1 - wealth / np.maximum.accumulate(wealth)
        return np.array([drawdown.max() if len(drawdown) else 0.0])
    windows = np.lib.stride_tricks.sliding_window_view(wealth, horizon)
    high_water = np.maximum.accumulate(windows, axis=1)
    return (1 - windows / high_water).max(axis=1)


def maximum_drawdown(simple_returns):
    wealth = np.cumprod(1 + np.asarray(simple_returns, dtype=float))
    drawdown = 1 - wealth / np.maximum.accumulate(wealth)
    return float(drawdown.max())


def moving_block_sample(values, length, block, rng):
    array = np.asarray(values, dtype=float)
    starts = rng.integers(0, len(array) - block + 1, size=int(np.ceil(length / block)))
    return np.concatenate([array[start:start + block] for start in starts])[:length]


def annual_pot_forecast(all_returns, held, probability):
    forecasts = pd.Series(index=held.index, dtype=float)
    records = []
    for year in sorted(held["Date"].dt.year.unique()):
        start = pd.Timestamp(f"{year}-01-01")
        history = all_returns[all_returns["Date"] < start]
        threshold = float(history["Loss (%)"].quantile(0.95))
        fit = fit_pot_tail_risk(history["Loss (%)"].to_numpy(), threshold=threshold, probability=probability)
        mask = held["Date"].dt.year == year
        forecasts.loc[held.index[mask]] = fit["value_at_risk"]
        records.append({
            "Forecast year": int(year),
            "History through": history["Date"].max(),
            "History observations": int(len(history)),
            "Threshold (%)": threshold,
            "Excesses": int(fit["exceedance_count"]),
            "Shape": fit["shape"],
            "Scale (%)": fit["scale"],
            "Forecast VaR (%)": fit["value_at_risk"],
            "Forecast ES (%)": fit["expected_shortfall"],
        })
    assert forecasts.notna().all()
    return forecasts, pd.DataFrame(records)

data = load_public_data()
returns, design, held = split_samples(data)
design_losses = design["Loss (%)"].to_numpy()
held_losses = held["Loss (%)"].to_numpy()
probability = 0.99
mu = float(design_losses.mean())
sigma = float(design_losses.std(ddof=1))
threshold = float(np.quantile(design_losses, 0.95))
pot = fit_pot_tail_risk(design_losses, threshold=threshold, probability=probability)
gaussian_forecast = np.full(len(held), mu + sigma * norm.ppf(probability))
empirical_forecast = np.full(len(held), np.quantile(design_losses, probability))
pot_forecast = np.full(len(held), pot["value_at_risk"])
annual_forecast, annual_records = annual_pot_forecast(returns, held, probability)
annual_values = annual_forecast.to_numpy()

forecast_definitions = pd.DataFrame([
    {"Forecast": "Gaussian", "Information used": "Design-sample loss mean and standard deviation", "Update rule": "Frozen"},
    {"Forecast": "Empirical", "Information used": "Design-sample loss quantile", "Update rule": "Frozen"},
    {"Forecast": "Generalized Pareto", "Information used": "Design-sample 95th-percentile excesses", "Update rule": "Frozen"},
    {"Forecast": "Annual generalized Pareto", "Information used": "All losses strictly before each forecast year", "Update rule": "Once at each year start"},
])
save_table(forecast_definitions, "forecast_definitions", "Quantile forecasts in the held-out comparison.", "tab:chapter22-forecast-definitions", fit=True)

frozen = pd.DataFrame([
    {"Forecast": "Gaussian", "99% VaR (%)": gaussian_forecast[0], "Varies through held-out period": False},
    {"Forecast": "Empirical", "99% VaR (%)": empirical_forecast[0], "Varies through held-out period": False},
    {"Forecast": "Generalized Pareto", "99% VaR (%)": pot_forecast[0], "Varies through held-out period": False},
    {"Forecast": "Annual generalized Pareto", "99% VaR (%)": annual_values.mean(), "Varies through held-out period": True},
])
save_table(frozen, "frozen_forecasts", "Forecast values fixed before or updated during the held-out period.", "tab:chapter22-frozen-forecasts")

forecast_frame = held[["Date", "Loss (%)"]].copy()
forecast_frame["Gaussian forecast (%)"] = gaussian_forecast
forecast_frame["Empirical forecast (%)"] = empirical_forecast
forecast_frame["Generalized Pareto forecast (%)"] = pot_forecast
forecast_frame["Annual generalized Pareto forecast (%)"] = annual_values
forecast_frame.to_csv(FIGURE_DATA / "heldout_loss_forecasts.csv", index=False)
figure, axis = plt.subplots(figsize=(11.4, 5.8), constrained_layout=True)
axis.plot(forecast_frame["Date"], forecast_frame["Loss (%)"], color=PALETTE["text"], linewidth=0.55, alpha=0.72, label="Held-out loss")
axis.plot(forecast_frame["Date"], forecast_frame["Gaussian forecast (%)"], color=PALETTE["amber"], linewidth=1.3, label="Gaussian")
axis.plot(forecast_frame["Date"], forecast_frame["Empirical forecast (%)"], color=PALETTE["green"], linewidth=1.3, label="Empirical")
axis.plot(forecast_frame["Date"], forecast_frame["Generalized Pareto forecast (%)"], color=PALETTE["cyan"], linewidth=1.5, label="Frozen generalized Pareto")
axis.plot(forecast_frame["Date"], forecast_frame["Annual generalized Pareto forecast (%)"], color=PALETTE["violet"], linewidth=1.4, label="Annual generalized Pareto")
axis.set_xlabel("Date")
axis.set_ylabel("Daily loss (%)")
axis.set_title("Held-Out Losses and 99% Quantile Forecasts")
axis.legend(ncol=2)
finish_axis(axis)
save_figure(figure, "heldout_loss_forecasts")
plt.close(figure)

zoom = forecast_frame[(forecast_frame["Date"] >= "2020-01-01") & (forecast_frame["Date"] <= "2020-12-31")].copy()
zoom.to_csv(FIGURE_DATA / "heldout_forecast_zoom.csv", index=False)
figure, axis = plt.subplots(figsize=(11.0, 5.5), constrained_layout=True)
axis.plot(zoom["Date"], zoom["Loss (%)"], color=PALETTE["text"], linewidth=0.8, label="Held-out loss")
axis.plot(zoom["Date"], zoom["Gaussian forecast (%)"], color=PALETTE["amber"], linewidth=1.4, label="Gaussian")
axis.plot(zoom["Date"], zoom["Empirical forecast (%)"], color=PALETTE["green"], linewidth=1.4, label="Empirical")
axis.plot(zoom["Date"], zoom["Generalized Pareto forecast (%)"], color=PALETTE["cyan"], linewidth=1.6, label="Frozen generalized Pareto")
axis.plot(zoom["Date"], zoom["Annual generalized Pareto forecast (%)"], color=PALETTE["violet"], linewidth=1.5, label="Annual generalized Pareto")
axis.set_xlabel("Date")
axis.set_ylabel("Daily loss (%)")
axis.set_title("Forecast Exceptions During 2020")
axis.legend(ncol=2)
finish_axis(axis)
save_figure(figure, "heldout_forecast_zoom")
plt.close(figure)

assert design["Date"].max() < held["Date"].min()
assert annual_records["History through"].dt.year.lt(annual_records["Forecast year"]).all()
assert np.isfinite(forecast_frame.drop(columns="Date").to_numpy()).all()

,Forecast,Information used,Update rule
0,Gaussian,Design-sample loss mean and standard deviation,Frozen
1,Empirical,Design-sample loss quantile,Frozen
2,Generalized Pareto,Design-sample 95th-percentile excesses,Frozen
3,Annual generalized Pareto,All losses strictly before each forecast year,Once at each year start


,Forecast,99% VaR (%),Varies through held-out period
0,Gaussian,3.686965,Fail
1,Empirical,4.4378,Fail
2,Generalized Pareto,4.481563,Fail
3,Annual generalized Pareto,4.374017,Pass


## Testing exception frequency and clustering

If \(N=\sum_{t=1}^T I_t\) exceptions occur among \(T\) forecasts, the observed exception proportion is \(\widehat a=N/T\). The likelihood ratio for the declared probability \(a\) is

$$
\operatorname{LR}_{\mathrm{cov}}=2\left[N\log\!\left(\frac{\widehat a}{a}\right)+(T-N)\log\!\left(\frac{1-\widehat a}{1-a}\right)\right].
$$

The statistic compares the likelihood under the observed exception rate with the likelihood under the declared rate. It checks frequency, not clustering. Its usual reference distribution has one degree of freedom and is an approximation when exceptions are few.

Let \(n_{ij}\) count transitions from yesterday's exception state \(i\) to today's state \(j\), where \(i,j\in\{0,1\}\). The state-dependent exception rates are

$$
\widehat\pi_i=\frac{n_{i1}}{n_{i0}+n_{i1}},
$$

and the pooled transition rate is

$$
\widehat\pi=\frac{n_{01}+n_{11}}{T-1}.
$$

The independence likelihood ratio compares the two state-dependent rates with the pooled rate. The combined statistic is

$$
\operatorname{LR}_{\mathrm{cc}}=\operatorname{LR}_{\mathrm{cov}}+\operatorname{LR}_{\mathrm{ind}}.
$$

The combined test has two asymptotic degrees of freedom. If either previous-state row has no transitions, the implemented independence and combined statistics are left unidentified rather than assigned an invented value.

In [2]:
data = load_public_data()
returns, design, held = split_samples(data)
design_losses = design["Loss (%)"].to_numpy()
held_losses = held["Loss (%)"].to_numpy()
probability = 0.99
tail_probability = 0.01
mu = design_losses.mean()
sigma = design_losses.std(ddof=1)
threshold = float(np.quantile(design_losses, 0.95))
pot = fit_pot_tail_risk(design_losses, threshold=threshold, probability=probability)
annual, annual_records = annual_pot_forecast(returns, held, probability)
forecasts = {
    "Gaussian": np.full(len(held), mu + sigma * norm.ppf(probability)),
    "Empirical": np.full(len(held), np.quantile(design_losses, probability)),
    "Generalized Pareto": np.full(len(held), pot["value_at_risk"]),
    "Annual generalized Pareto": annual.to_numpy(),
}

summary_rows = []
test_rows = []
transition_rows = []
cumulative = pd.DataFrame({"Date": held["Date"].to_numpy(), "Expected exceptions": np.arange(1, len(held) + 1) * tail_probability})
for name, forecast in forecasts.items():
    exceptions = (held_losses > forecast).astype(int)
    result = quantile_exception_tests(exceptions, exception_probability=tail_probability)
    longest = max((len(group) for group in "".join(map(str, exceptions)).split("0") if group), default=0)
    summary_rows.append({"Forecast": name, "Held-out observations": len(exceptions), "Expected exceptions": len(exceptions) * tail_probability, "Observed exceptions": result["exception_count"], "Observed rate": result["exception_count"] / len(exceptions), "Longest exception run": longest})
    test_rows.append({"Forecast": name, "Coverage statistic": result["coverage_statistic"], "Coverage p-value": result["coverage_pvalue"], "Independence statistic": result["independence_statistic"], "Independence p-value": result["independence_pvalue"], "Combined statistic": result["conditional_coverage_statistic"], "Combined p-value": result["conditional_coverage_pvalue"]})
    transition_rows.append({"Forecast": name, "0 to 0": result["transitions"][0, 0], "0 to 1": result["transitions"][0, 1], "1 to 0": result["transitions"][1, 0], "1 to 1": result["transitions"][1, 1]})
    cumulative[f"{name} cumulative exceptions"] = np.cumsum(exceptions)

exception_summary = pd.DataFrame(summary_rows)
exception_tests = pd.DataFrame(test_rows)
transition_counts = pd.DataFrame(transition_rows)
save_table(exception_summary, "exception_summary", "Exception counts for the 99\\% held-out forecasts.", "tab:chapter22-exception-summary", fit=True)
save_table(exception_tests, "exception_tests", "Coverage, independence, and combined tests at the 1\\% tail probability.", "tab:chapter22-exception-tests", fit=True)
save_table(transition_counts, "transition_counts", "Transitions between exception states.", "tab:chapter22-transition-counts")

cumulative.to_csv(FIGURE_DATA / "cumulative_exceptions.csv", index=False)
figure, axis = plt.subplots(figsize=(11.2, 5.6), constrained_layout=True)
axis.plot(cumulative["Date"], cumulative["Expected exceptions"], color=PALETTE["text"], linestyle="--", linewidth=1.5, label="Expected under 1% rate")
colors = {"Gaussian": PALETTE["amber"], "Empirical": PALETTE["green"], "Generalized Pareto": PALETTE["cyan"], "Annual generalized Pareto": PALETTE["violet"]}
for name, color in colors.items():
    axis.step(cumulative["Date"], cumulative[f"{name} cumulative exceptions"], where="post", color=color, linewidth=1.5, label=name)
axis.set_xlabel("Date")
axis.set_ylabel("Cumulative exceptions")
axis.set_title("Cumulative Observed and Expected Exceptions")
axis.legend(ncol=2)
finish_axis(axis)
save_figure(figure, "cumulative_exceptions")
plt.close(figure)

pvalues = exception_tests.melt(id_vars="Forecast", value_vars=["Coverage p-value", "Independence p-value", "Combined p-value"], var_name="Test", value_name="p-value")
pvalues.to_csv(FIGURE_DATA / "backtest_pvalue_heatmap.csv", index=False)
matrix = pvalues.pivot(index="Forecast", columns="Test", values="p-value").reindex(index=list(forecasts), columns=["Coverage p-value", "Independence p-value", "Combined p-value"])
figure, axis = plt.subplots(figsize=(9.0, 5.8), constrained_layout=True)
image = axis.imshow(matrix.to_numpy(), vmin=0, vmax=1, cmap="viridis")
axis.set_xticks(range(3), ["Coverage", "Independence", "Combined"])
axis.set_yticks(range(len(matrix.index)), matrix.index)
axis.set_title("Held-Out Quantile Test p-Values")
for i in range(len(matrix.index)):
    for j in range(3):
        value = matrix.iloc[i, j]
        axis.text(j, i, f"{value:.3f}", ha="center", va="center", color="white" if value < 0.55 else PALETTE["background"])
colorbar = figure.colorbar(image, ax=axis)
colorbar.set_label("p-value")
save_figure(figure, "backtest_pvalue_heatmap")
plt.close(figure)

pot_row = exception_summary[exception_summary["Forecast"] == "Generalized Pareto"].iloc[0]
pot_tests = exception_tests[exception_tests["Forecast"] == "Generalized Pareto"].iloc[0]
assert int(pot_row["Observed exceptions"]) == 16
assert pot_tests["Coverage p-value"] < 0.05
assert pot_tests["Independence p-value"] < 0.05
assert pot_tests["Combined p-value"] < 0.05

,Forecast,Held-out observations,Expected exceptions,Observed exceptions,Observed rate,Longest exception run
0,Gaussian,"2,767",27.67,38,0.013733,2
1,Empirical,"2,767",27.67,16,0.005782,2
2,Generalized Pareto,"2,767",27.67,16,0.005782,2
3,Annual generalized Pareto,"2,767",27.67,18,0.006505,2


,Forecast,Coverage statistic,Coverage p-value,Independence statistic,Independence p-value,Combined statistic,Combined p-value
0,Gaussian,3.489043,0.061776,2.535577,0.111306,6.024621,0.049178
1,Empirical,5.861323,0.015477,8.955851,0.002766,14.817174,0.000606
2,Generalized Pareto,5.861323,0.015477,8.955851,0.002766,14.817174,0.000606
3,Annual generalized Pareto,3.894922,0.048432,7.997002,0.004685,11.891925,0.002616


,Forecast,0 to 0,0 to 1,1 to 0,1 to 1
0,Gaussian,"2,692",36,36,2
1,Empirical,"2,736",14,14,2
2,Generalized Pareto,"2,736",14,14,2
3,Annual generalized Pareto,"2,732",16,16,2


## Testing annual updates without future data

For forecast year \(y\), the updated threshold is

$$
u_y=Q_{0.95}\bigl(\{L_t:t<\text{January 1 of year }y\}\bigr).
$$

The function \(Q_{0.95}\) is the empirical 95th percentile and the set contains only losses observed before year \(y\). A generalized Pareto fit using these past losses produces one forecast for that calendar year. Updating preserves the time order, but it does not guarantee correct exception frequency or independence.

At tail probability \(a\), the expected exception count over \(T\) forecasts is

$$
\mathbb{E}[N]=Ta.
$$

The count \(N\) is random even under a correct forecast. Comparing observed and expected counts at several declared levels is a sensitivity check. It does not justify selecting a level after observing which test passes.

In [3]:
data = load_public_data()
returns, design, held = split_samples(data)
design_losses = design["Loss (%)"].to_numpy()
held_losses = held["Loss (%)"].to_numpy()
annual_main, annual_records = annual_pot_forecast(returns, held, 0.99)
save_table(annual_records, "annual_pot_updates", "Past-only annual generalized Pareto updates.", "tab:chapter22-annual-updates", fit=True)

annual_plot = annual_records[["Forecast year", "Threshold (%)", "Shape", "Scale (%)", "Forecast VaR (%)"]].copy()
annual_plot.to_csv(FIGURE_DATA / "annual_pot_updates.csv", index=False)
figure, axes = plt.subplots(2, 1, figsize=(10.8, 7.0), sharex=True, constrained_layout=True)
axes[0].plot(annual_plot["Forecast year"], annual_plot["Threshold (%)"], color=PALETTE["amber"], marker="o", linewidth=1.6, label="Threshold")
axes[0].plot(annual_plot["Forecast year"], annual_plot["Forecast VaR (%)"], color=PALETTE["cyan"], marker="s", linewidth=1.6, label="99% forecast")
axes[0].set_ylabel("Loss (%)")
axes[0].set_title("Past-Only Annual Tail Updates")
axes[0].legend()
axes[1].plot(annual_plot["Forecast year"], annual_plot["Shape"], color=PALETTE["violet"], marker="o", linewidth=1.6, label="Shape")
axes[1].plot(annual_plot["Forecast year"], annual_plot["Scale (%)"], color=PALETTE["green"], marker="s", linewidth=1.6, label="Scale (%)")
axes[1].set_xlabel("Forecast year")
axes[1].set_ylabel("Fitted parameter")
axes[1].legend()
for axis in axes:
    finish_axis(axis)
save_figure(figure, "annual_pot_updates")
plt.close(figure)

sensitivity_rows = []
figure_rows = []
mu = design_losses.mean()
sigma = design_losses.std(ddof=1)
for probability in [0.975, 0.99, 0.995]:
    tail_probability = 1 - probability
    threshold = float(np.quantile(design_losses, 0.95))
    pot = fit_pot_tail_risk(design_losses, threshold=threshold, probability=probability)
    annual, records = annual_pot_forecast(returns, held, probability)
    methods = {
        "Gaussian": np.full(len(held), mu + sigma * norm.ppf(probability)),
        "Empirical": np.full(len(held), np.quantile(design_losses, probability)),
        "Generalized Pareto": np.full(len(held), pot["value_at_risk"]),
        "Annual generalized Pareto": annual.to_numpy(),
    }
    for name, forecast in methods.items():
        exceptions = (held_losses > forecast).astype(int)
        result = quantile_exception_tests(exceptions, exception_probability=tail_probability)
        sensitivity_rows.append({"Probability level": probability, "Forecast": name, "Expected exceptions": len(held) * tail_probability, "Observed exceptions": result["exception_count"], "Coverage p-value": result["coverage_pvalue"], "Independence p-value": result["independence_pvalue"], "Combined p-value": result["conditional_coverage_pvalue"]})
        figure_rows.append({"Probability level": probability, "Forecast": name, "Observed divided by expected": result["exception_count"] / (len(held) * tail_probability)})
sensitivity = pd.DataFrame(sensitivity_rows)
save_table(sensitivity, "sensitivity_backtests", "Held-out exception tests across three probability levels.", "tab:chapter22-backtest-sensitivity", fit=True)

coverage = pd.DataFrame(figure_rows)
coverage.to_csv(FIGURE_DATA / "coverage_sensitivity.csv", index=False)
figure, axis = plt.subplots(figsize=(10.8, 5.6), constrained_layout=True)
colors = {"Gaussian": PALETTE["amber"], "Empirical": PALETTE["green"], "Generalized Pareto": PALETTE["cyan"], "Annual generalized Pareto": PALETTE["violet"]}
for name, part in coverage.groupby("Forecast"):
    axis.plot(part["Probability level"], part["Observed divided by expected"], color=colors[name], marker="o", linewidth=1.7, label=name)
axis.axhline(1, color=PALETTE["text"], linestyle="--", linewidth=1.2, label="Observed equals expected")
axis.set_xlabel("Probability level")
axis.set_ylabel("Observed exceptions divided by expected")
axis.set_title("Exception Count Sensitivity")
axis.legend(ncol=2)
finish_axis(axis)
save_figure(figure, "coverage_sensitivity")
plt.close(figure)

alternating = quantile_exception_tests(np.tile([0, 1], 20), exception_probability=0.5)
no_exceptions = quantile_exception_tests(np.zeros(40, dtype=int), exception_probability=0.01)
edge_checks = pd.DataFrame([
    {"Input": "Alternating zero and one", "Coverage identified": alternating["coverage_statistic"] is not None, "Independence identified": alternating["independence_statistic"] is not None},
    {"Input": "No exceptions", "Coverage identified": no_exceptions["coverage_statistic"] is not None, "Independence identified": no_exceptions["independence_statistic"] is not None},
])
save_table(edge_checks, "library_edge_case_checks", "Identification behavior for two exception sequences.", "tab:chapter22-edge-cases")

limits = pd.DataFrame([
    {"Result": "Small p-value for coverage", "Permitted reading": "Observed exception frequency conflicts with the declared rate", "Excluded reading": "Expected shortfall is invalid"},
    {"Result": "Small p-value for independence", "Permitted reading": "Exceptions show first-order clustering", "Excluded reading": "A unique economic cause is proven"},
    {"Result": "Large p-value", "Permitted reading": "The test does not reject in this sample", "Excluded reading": "The forecast model is true"},
    {"Result": "Updated forecast", "Permitted reading": "Only past observations enter each update", "Excluded reading": "Updating guarantees calibration"},
])
save_table(limits, "backtest_interpretation_limits", "Interpretation limits for quantile exception tests.", "tab:chapter22-backtest-limits", fit=True)

assert annual_records["History through"].dt.year.lt(annual_records["Forecast year"]).all()
assert len(sensitivity) == 12
assert no_exceptions["independence_statistic"] is None

,Forecast year,History through,History observations,Threshold (%),Excesses,Shape,Scale (%),Forecast VaR (%),Forecast ES (%)
0,"2,015",2014-12-31,"5,287",2.614336,265,0.135989,1.036079,4.481563,5.974601
1,"2,016",2015-12-31,"5,539",2.57528,277,0.119066,1.054294,4.445815,5.895425
2,"2,017",2016-12-30,"5,791",2.550811,290,0.121619,1.037318,4.396865,5.833411
3,"2,018",2017-12-29,"6,042",2.510569,303,0.127522,1.020848,4.338044,5.775203
4,"2,019",2018-12-31,"6,293",2.50937,315,0.117604,1.023326,4.323909,5.725459
5,"2,020",2019-12-31,"6,546",2.4877,328,0.117565,1.01157,4.28259,5.66806
6,"2,021",2020-12-31,"6,799",2.510707,340,0.135411,1.059291,4.415804,5.939376
7,"2,022",2021-12-31,"7,051",2.505419,353,0.157706,1.005556,4.349378,5.888459
8,"2,023",2022-12-30,"7,302",2.5441,366,0.131942,1.035146,4.40336,5.878446
9,"2,024",2023-12-29,"7,552",2.509939,378,0.13135,1.03212,4.36106,5.829161


,Probability level,Forecast,Expected exceptions,Observed exceptions,Coverage p-value,Independence p-value,Combined p-value
0,0.975,Gaussian,69.175,59,0.203891,0.044555,0.059318
1,0.975,Empirical,69.175,51,0.020355,0.01559,0.003643
2,0.975,Generalized Pareto,69.175,50,0.01411,0.013446,0.002318
3,0.975,Annual generalized Pareto,69.175,53,0.040093,0.020713,0.008376
4,0.99,Gaussian,27.67,38,0.061776,0.111306,0.049178
5,0.99,Empirical,27.67,16,0.015477,0.002766,0.000606
6,0.99,Generalized Pareto,27.67,16,0.015477,0.002766,0.000606
7,0.99,Annual generalized Pareto,27.67,18,0.048432,0.004685,0.002616
8,0.995,Gaussian,13.835,23,0.024175,0.013767,0.003792
9,0.995,Empirical,13.835,5,0.006099,0.005072,0.000459


,Input,Coverage identified,Independence identified
0,Alternating zero and one,Pass,Pass
1,No exceptions,Pass,Fail


,Result,Permitted reading,Excluded reading
0,Small p-value for coverage,Observed exception frequency conflicts with th...,Expected shortfall is invalid
1,Small p-value for independence,Exceptions show first-order clustering,A unique economic cause is proven
2,Large p-value,The test does not reject in this sample,The forecast model is true
3,Updated forecast,Only past observations enter each update,Updating guarantees calibration
